In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] ='0'
from huggingface_hub import notebook_login

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


- log in Hugging Face 

In [2]:
notebook_login()

# Phase 1: System Design & Stack Selection

- __Language & Framework:__ Python with LangChain. It provides a strong set of core components for building both RAG systems and tool-based agents.
- __Data Ingestion:__ WebBaseLoader for the SAGES and AME HTML links, and PyPDFLoader for the ERAS booklet.
- __Chunking Strategy:__ RecursiveCharacterTextSplitter. Since surgical steps and guidelines are highly sequential, overlapping chunks (e.g., 1000 tokens with a 200-token overlap) will help maintain context across operative steps.
- __Vector Store:__ ChromaDB or FAISS. Both are lightweight, run locally, and require zero external infrastructure, which is perfect for a take-home CLI tool.
- __Agent Paradigm:__ A Tool-Calling Agent. We will wrap your RAG pipeline into a "Clinical Knowledge Retriever" tool. We will also implement Structured Output using Pydantic, requiring the LLM to format its response with specific fields (e.g., current_step, next_action, safety_warnings) to satisfy the agent capability requirement.

### Here is a clean Python script using LangChain to ingest the web and PDF sources, along with a strategic chunking configuration.

# Phase 2: Data Ingestion & Chunking Strategy

In [3]:
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

USER_AGENT environment variable not set, consider setting it to identify your requests.


### __Ingests clinical guidelines from URLs and PDFs, then chunks them for optimal Vector Store retrieval.__

## 1. Define our source documents

In [4]:
sages_url = "https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/"
ame_url = "https://ales.amegroups.org/article/view/5766/html"
eras_pdf_url = "https://www.urmc.rochester.edu/getmedia/c6bc9e17-c349-436c-926e-bf4f4e498d8d/ERAS-Cholecystectomy-Booklet.pdf"

### Load HTML Web Documents (SAGES & AME)

At a high level, *WebBaseLoader* in LangChain is a tool that handles the process of pulling HTML from a URL and turning it into a clean, usable document object.

I used it to load sources like the SAGES guidelines and the AME Operative Technique article. While I could have written a custom BeautifulSoup script, the advantage of *WebBaseLoader* is that it standardizes the output for the rest of the LangChain pipeline.

Most importantly, it automatically keeps the source URL as part of the document’s metadata. In a clinical workflow, knowing where the information comes from is important; if the agent suggests a specific surgical technique, we need to link that recommendation back to the exact SAGES source.

In [5]:
web_loader = WebBaseLoader(web_paths=[sages_url, ame_url])
web_docs = web_loader.load()

In [ ]:
web_docs[0]

### Load PDF Document (ERAS)

Similar to the *WebBaseLoader*, *PyPDFLoader* is LangChain’s built-in tool for working with PDF files. I used it to process the ERAS Cholecystectomy booklet.

The main advantage is how it handles document structure. Instead of loading the entire PDF as one large block of text, it automatically splits the content page by page and includes the page number in the metadata for each section.

This is very useful during retrieval. If the agent pulls a safety protocol from the ERAS booklet, we can immediately see which page it came from. It makes debugging the pipeline much easier and helps keep the model’s responses clearly connected to the source.

In [6]:
pdf_loader = PyPDFLoader(eras_pdf_url)
pdf_docs = pdf_loader.load()

In [ ]:
pdf_docs

In [7]:
len(pdf_docs)

12

### Combine all raw documents

In [8]:
all_raw_docs = web_docs + pdf_docs
print(f"Successfully loaded {len(all_raw_docs)} raw documents.")

Successfully loaded 14 raw documents.


### Chunking Strategy

Chunking means taking a large document and splitting it into smaller sections before storing it in a vector database.

We do this for two main reasons. First, LLMs have limits on how much text they can process at once, so we can’t include an entire textbook in a single prompt.

More importantly, it improves search accuracy. If I embed a 50-page surgical manual as one vector, its mathematical meaning becomes too averaged out. But if I split it into smaller sections, the model can retrieve the exact part that’s relevant.

For example, if a user asks about Calot’s triangle, we want the system to return the specific paragraph that explains it, not the entire manual. Chunking makes that retrieval much more precise.

__I chose a chunk_size of 1000 tokens with a chunk_overlap of 200. This is a standard "Goldilocks" zone for medical texts; large enough to capture an entire concept (like the criteria for the Critical View of Safety) but overlapping enough so that continuous operative steps aren't artificially severed.__

*RecursiveCharacterTextSplitter* is a specific chunking method in LangChain, and it’s one of the most effective options for working with standard text. Instead of splitting the text at fixed intervals, like every 500 words, which can break sentences or surgical steps, it uses a step-by-step approach.

It first tries to split by double newlines to keep paragraphs intact. If a section is still too large, it then splits by single newlines to preserve sentence structure, and continues in that way.

I chose this method because clinical guidelines are highly structured, with headings, bullet points, and ordered steps. The key advantage is that it follows the natural structure of the human language. This helps ensure that important details, like safety warnings, stay connected to the surgical steps they relate to.

- The __chunk_size__ is simply the maximum number of characters in each text segment. For this pipeline, I set it to 1000 characters. In practice, this works well for medical text; it’s large enough to cover an entire concept, like the full criteria for the Critical View of Safety, but still small enough to keep retrieval accurate.

- The __chunk_overlap__ defines how much text is shared between neighboring segments. I set it to 200 characters, meaning the last part of one segment is repeated at the start of the next. This is especially important for surgical workflows, where steps follow a clear sequence. If one step ends at the boundary and the next begins in a new segment, the overlap helps preserve that connection so the model doesn’t lose important context.

In [9]:
# Using RecursiveCharacterTextSplitter to split by paragraphs/sentences to preserve clinical context boundaries.

text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        add_start_index=True, # Helps track where in the document the chunk came from
        separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)

chunked_docs = text_splitter.split_documents(all_raw_docs)
    
print(f"Split documents into {len(chunked_docs)} manageable chunks.")

Split documents into 274 manageable chunks.


In [12]:
print("\n--- Preview of Chunk 1 ---")
print(f"Source: {chunked_docs[2].metadata.get('source')}")
print(f"Content: {chunked_docs[2].page_content[:300]}...\n")


--- Preview of Chunk 1 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content: Member Spotlight
Give the Gift of SAGES Membership


Patients

Join the SAGES Patient Partner Network (PPN)
Patient Information Brochures
Healthy Sooner – Patient Information for Minimally Invasive Surgery
Choosing Wisely – An Initiative of the ABIM Foundation
All in the Recovery: Colorectal Cancer ...



__Unified Pipeline: It seamlessly handles both standard web scraping and PDF parsing in one go.__

__Metadata Preservation: The LangChain loaders automatically attach the source URLs to the metadata of each chunk. This is critical for RAG, as my agent will eventually need to cite exactly which document it pulled the surgical step from.__

Now that we have our text neatly divided into overlapping clinical chunks, we need to embed them into a Vector Store so our model can query them.

# Phase 3: The RAG Pipeline (Embedding & Retrieval)

__Reason for Embedding:__
You can’t just pass raw text into an AI system and expect it to understand how clinical concepts relate to each other. Embedding acts as a translation layer. It takes our human-readable sections of surgical guidelines and converts them into numerical vectors.

We do this because a simple keyword search isn’t reliable enough for medical data. For example, if a user asks about ‘removing the gallbladder,’ but the document uses the term ‘cholecystectomy,’ a keyword search might miss it.

With embeddings, the model captures the meaning behind the words, so those two terms are represented very closely in vector space. This is what allows the system to retrieve the most relevant information in a smart and flexible way.

Standard top-k retrieval might pull four chunks that all describe the exact same sentence from different angles. MMR fetches a larger pool of chunks and then selects the most diverse set, ensuring the LLM gets a broader context of the surgical procedure (e.g., pulling a chunk about Calot's triangle dissection and a chunk about the safety risks).

For the embedding model, *BAAI/bge-small-en-v1.5* (via Hugging Face) is the best lightweight embedding model for retrieval tasks. It runs fast and has excellent semantic understanding of __medical texts__. For the vector database, ChromaDB is perfect because it stores the database locally as a file, requiring no complex server setup for the live demo.

## 3.1 Initializing Local Embeddings
To maintain strict data privacy suitable for hospital environments, we utilize a local, open-weights embedding model (`BAAI/bge-small-en-v1.5`). We are accelerating this process using GPU compute.

In [10]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

__What & Why is HuggingFaceEmbeddings?__

It’s basically LangChain’s connection to the Hugging Face ecosystem. Instead of writing custom PyTorch code to load a model, tokenize text, pool hidden states, and manage device placement, HuggingFaceEmbeddings wraps all of that into a simple, easy-to-use class.

I chose it here because it supports running everything locally. By setting the device to ‘cuda,’ I could run the embedding step directly on the V100 GPU in our HPC environment, keeping the pipeline private and very fast.

__What & Why is BAAI/bge-small-en-v1.5?__

BGE-small is an open-weight embedding model developed by the Beijing Academy of Artificial Intelligence. I chose it because it performs very strongly on benchmarks like MTEB (Massive Text Embedding Benchmark).

The key advantage of this clinical retrieval task is efficiency. It’s a smaller model, producing 384-dimensional vectors compared to the larger 1536-dimensional vectors from models like OpenAI’s.

Since I’m running everything locally, I don’t need a large model to achieve accurate semantic search. This model provides a strong balance. It captures detailed medical meaning while remaining fast enough that the user experiences near real-time performance.

__Additional Note__

BGE-small is a general-purpose embedding model, rather than a medical-specific model like ClinicalBERT or BioBERT, which are trained mainly on clinical notes and PubMed articles. I chose it intentionally because the field has evolved quite a bit.

Newer general models like BGE are trained on very large and diverse datasets, including a significant amount of scientific and medical literature. Even though the data isn’t only medical, the overall amount of medical content it has seen is much larger than what older specialized models were trained on.

Another key point is how it’s trained. BGE uses contrastive learning on pairs of text, such as questions and answers. This makes it especially good at connecting clinical terminology in guidelines with the natural language a user might type.

For a lightweight pipeline running on a single GPU, BGE-small offers a strong balance. It captures medical meaning accurately while staying fast enough for real-time use, without relying on a large, resource-heavy model.

In [11]:
print("Initializing local HuggingFace embeddings...")

# Initialize Local Embeddings
# Pushing the model to the GPU for near-instantaneous embedding generation
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cuda:0'} 
)

print("Embeddings loaded successfully on GPU.")

Initializing local HuggingFace embeddings...
Embeddings loaded successfully on GPU.


## 3.2 Building the Vector Database

__Purpose of building the Vector Database:__

If embeddings are like GPS coordinates for our clinical data, then the vector database is the map.

After converting sections of the SAGES and ERAS guidelines into numerical vectors, we need an efficient system to store and search them. When a user asks a question, we convert that question into a vector as well, and the database compares it to all stored vectors using cosine similarity, which measures how close they are in meaning.

It then returns the most relevant sections based on that similarity. Without a vector database, the retrieval step in RAG wouldn’t be possible.

__Why did we choose langchain_community.vectorstores.Chroma?__

I chose Chroma because it’s lightweight and works well for a privacy-focused setup.

In a clinical setting, I wanted to avoid cloud-based vector databases, since sending hospital data to external servers conflicts with a local, secure design. At the same time, I didn’t want the added complexity of setting up a full-scale database system just for a demo.

Chroma runs locally within the Python environment, and its persist_directory feature allows me to save the database directly to disk. I can build the index once and then load it quickly for a live demo, without waiting for it to rebuild each time.

__ChromaDB__ is an open-source vector database. If embeddings are like coordinates, Chroma is the system that stores and organizes them.

I chose it because it’s lightweight and easy to work with. In a cloud production setup, you might use something like AWS OpenSearch. But for a clinical, local setup, and especially for a live demo, Chroma works really well.

The key advantage is that it runs locally and can save data directly to a disk folder. I didn’t need to set up containers or rely on external APIs. I could embed the SAGES and ERAS guidelines once, save them, and then load them quickly without network delays or data privacy concerns.

We use ChromaDB to store our chunked clinical guidelines locally. The database is persisted to disk so it can be instantly reloaded in future sessions without re-embedding.

In [12]:
persist_dir = "./chroma_db_clinical"
print("Building and persisting local Vector Store. This may take a moment...")

vector_store = Chroma.from_documents(
    documents=chunked_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

print(f"Successfully embedded chunks into local ChromaDB at {persist_dir}.")

Building and persisting local Vector Store. This may take a moment...
Successfully embedded chunks into local ChromaDB at ./chroma_db_clinical.


## 3.3 Configuring the Clinical Retriever

__Retriever__

The vector database mainly stores numerical data; it doesn’t actively perform the search by itself. The retriever is the component that handles the search process.

We need this step because the LLM and the database work in different formats. When a user asks a question like, ‘What are the risks of dissecting Calot’s triangle?’, the retriever takes that text, converts it into a vector using the embedding model, and runs a similarity search against ChromaDB.

It then converts the results back into readable sections of text. In that sense, it connects the user’s question with the relevant information stored in the database.

Standard retrieval often pulls highly redundant chunks. To ensure our agent receives a comprehensive view of the surgical step and associated safety risks, we configure the retriever to use Maximal Marginal Relevance (MMR).

__Maximal Marginal Relevance (MMR)__

Standard vector search focuses on the highest similarity score. The issue is that if I ask about Calot’s triangle, it might return several paragraphs from the same document that repeat the same idea in slightly different ways.

Maximal Marginal Relevance, or MMR, addresses this by balancing relevance with diversity. It selects results that are closely related to the query, but it penalizes them if they are too similar to the chunks it has already selected.

I used this approach for the clinical retriever because surgeons need comprehensive context, not repeated information. For example, if they ask about a dissection step, MMR can return one section on anatomy, another on the tools involved, and another that explains the safety risks. This gives the LLM a more complete set of information to generate a high-quality answer.

__Used Parameter:__

- `k`: sets how many text sections we pass to the LLM. We need this because the model has a limited context window, so we have to be careful about how much information we include. If we send too much text, the model can lose focus or miss important details. I chose `k = 4` because, with a chunk size of about 1000 tokens, it provides around 4,000 tokens of context. In my experience, this works well—it gives enough detail for complex clinical questions while still leaving room for the prompt and the model’s response within the 8k limit.

- `fetch_k`: The `fetch_k` parameter works together with MMR. It defines how many candidate text sections we consider before selecting the final set. If we only looked at 4 text sections from the start, MMR wouldn’t have enough options to choose from if those results were too similar. By setting `fetch_k = 15`, we first identify the top 15 most relevant sections. Then MMR reviews those and selects the best 4 while keeping the results diverse. I chose 15 because it’s a practical and effective setting; it gives a broad enough set to capture different aspects of a surgical topic, while still being fast enough to keep the system responsive during a live demo.

In [13]:
# Configure the Retriever with MMR
retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,         # Return exactly 4 distinct chunks to the LLM
        "fetch_k": 15   # Initially fetch 15 similar chunks to evaluate for diversity
    }
)

print("Retriever configured with MMR (k=4, fetch_k=15).")

Retriever configured with MMR (k=4, fetch_k=15).


When the user asks a question, first, it pushes the user's question through the `BGE-small` embedding model to turn it into a vector. Then, the __Retriever__ steps in. It scans the ChromaDB vector database and calculates the mathematical distance between the question's vector and the vectors of all our document chunks. I specifically configured it to use __MMR__ to retrieve the top 4 most relevant, yet diverse, chunks of clinical guidelines.

## 3.4 Testing the Retrieval Pipeline
Let's verify the pipeline retrieves accurate and diverse context for a critical safety concept in laparoscopic cholecystectomy.

In [15]:
# Test query relevant to the clinical guidelines
test_query = "What is the Critical View of Safety (CVS)?"
print(f"Querying Vector Store: '{test_query}'\n")

retrieved_docs = retriever.invoke(test_query)

# Display the source and content of the top retrieved chunks
for i, doc in enumerate(retrieved_docs):
    print(f"--- Retrieved Chunk {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Content snippet: {doc.page_content[:300]}...\n")

Querying Vector Store: 'What is the Critical View of Safety (CVS)?'

--- Retrieved Chunk 1 ---
Source: https://ales.amegroups.org/article/view/5766/html
Content snippet: Step 2: Establishing the critical view of safety...

--- Retrieved Chunk 2 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: GUIDELINE RECOMMENDATIONS:
Question 1: Should the critical view of safety (CVS) versus other techniques (e.g. infundibular, top down, or intraoperative cholangiography) be used to mitigate the risk of bile duct injury during laparoscopic cholecystectomy?
Recommendation: In patients undergoing laparo...

--- Retrieved Chunk 3 ---
Source: https://www.sages.org/publications/guidelines/safe-cholecystectomy-multi-society-practice-guideline/
Content snippet: Narrative synthesis: Forty-five full text articles identified by the search methodology were reviewed that included three systematic reviews.
Use of Critical View of Sa

# Phase 4: Agent Construction & Structured Reasoning

## 4.1 Loading the Inference Engine (vLLM)
Instead of relying on external APIs, we load Llama-3 8B directly into the V100 GPU's memory using vLLM. This ensures maximum privacy for clinical environments while providing high-throughput token generation.

__vLLM__ is an open-source inference algorithm designed to run large language models fast and efficiently. If I used a standard Hugging Face `pipeline`, it would be slower because traditional Transformers don’t manage memory very efficiently during text generation.

vLLM addresses this with a method called `PagedAttention`. It manages the model’s memory (the KV cache) in smaller segments, similar to how an operating system handles memory. This greatly reduces memory waste and improves performance.

I chose it because, in a clinical setting, the system needs to respond in real time during a procedure. vLLM provides that speed while maintaining accuracy.

- __High Performance:__ It makes better use of the GPU through techniques like continuous batching, which processes requests in real time and reduces waiting periods.
- __PagedAttention:__ This approach is based on how operating systems handle memory. It divides memory into smaller sections, which greatly reduces fragmentation in the KV cache and allows the model to handle larger batch sizes more efficiently.
- __Ease of Use:__ It integrates smoothly with Hugging Face models and offers straightforward Python APIs, an OpenAI-compatible server, and support for different types of hardware.
- __Wide Compatibility:__ It supports a wide range of popular models, like LLaMA, Mistral, and Mixtral, and includes features like tensor parallelism for efficient multi-GPU performance.

__What is PagedAttention in vLLM?__

Think about how an operating system manages RAM. In older LLM setups, like standard Hugging Face pipelines, the system tries to reserve memory for both the prompt and the generated output all at once, in one large continuous block. Since it doesn’t know how long the output will be, it often reserves more memory than needed, which leads to a lot of wasted space—sometimes over 50% of the available VRAM.

PagedAttention addresses this by following a similar idea to virtual memory in operating systems. Instead of requiring one large block, it divides memory into smaller, fixed-size sections called pages. As Llama-3 generates tokens step by step, memory is assigned page by page as needed. This greatly reduces memory waste.

This is the key idea behind why vLLM can deliver high speed and efficiency, even on a single V100 GPU.

__What is KV-cache in vLLM?__

The KV-cache, or Key-Value cache, is essentially the model’s short-term memory during text generation.

Since LLMs generate text step by step  (one word at a time), they would normally need to recalculate the attention matrices for the entire previous context each time they generate a new token. For example, with a 1000-token clinical context, recalculating all 1000 tokens just to produce the next one would be very slow and costly.

To address this, the model computes the ‘Key’ and ‘Value’ tensors once and stores them in the cache. When generating the next step—like the next part of a surgical procedure—it only needs to compute the new token and can reuse the stored context from the KV-cache.

The challenge is that this cache grows quickly, which is why vLLM uses PagedAttention to manage it efficiently and prevent the GPU from running out of memory during longer sequences.

__Why did you choose `meta-llama/Meta-Llama-3-8B-Instruct`?__

This choice was based on balancing hardware limits and model alignment. The 8-billion parameter size works well here—it fits smoothly within the 32GB of VRAM on a single V100 GPU, while still leaving space for the embedding model.

The key factor is the Instruct version. The base Llama-3 model mainly predicts the next token, so if you ask a question, it may just generate more questions instead of giving a clear answer. The Instruct model has already been trained with methods like Supervised Fine-Tuning and Direct Preference Optimization, so it knows how to follow system prompts and respond in a structured way.

I needed a model that could reliably follow the Pydantic format and produce clean JSON outputs every time, and the Instruct version does that without additional setup.

__Why set `max_new_tokens=1024`?__

That parameter acts as a safety limit. It defines the maximum number of tokens the model can generate for a single response.

I set it to 1024 to match our JSON structure. For complex questions, the model needs enough space to produce the full response, including the JSON fields, the primary answer, and any safety notes. 1024 tokens give more than enough room for a detailed clinical answer.

At the same time, it sets a clear upper limit on computation. If the model ever gets stuck or starts repeating itself, it will stop at 1024 tokens instead of continuing and using up the GPU.

__1024 tokens is around how many words?__

As a general rule of thumb for standard English text, 1 token is roughly equal to ¾ of a word (or 100 tokens = 75 words). Therefore, setting `max_new_tokens=1024` gives the model a limit of roughly 750 to 800 words.

__What is the temperature parameter?__

Temperature controls how much randomness the model uses when choosing the next word. The model assigns probabilities to many possible next words, and a higher temperature flattens those probabilities, allowing for more unexpected choices.

In my setup, I set the temperature to `0.0`. This makes the model always choose the most likely next word. For creative tasks, a higher temperature can be useful. But for a clinical assistant, especially when interpreting something like the Critical View of Safety, we want consistency and accuracy.

Setting the temperature to `0.0` ensures the model produces stable, reliable responses that stay closely aligned with the clinical context.

__Why set `gpu_memory_utilization = 0.8`?__

By default, vLLM tries to reserve most of the available VRAM to maximize its PagedAttention KV-cache size. If left unchanged, it can take over 90% of the GPU memory.

The challenge is that I’m also running the `BGE-small` embedding model on the same GPU. If vLLM uses all available memory, the embedding model will fail when processing a user query and crash the whole RAG pipeline.

By setting the memory usage to `0.8`, I’m dividing the GPU resources so that vLLM uses 80% of the VRAM, while the remaining 20% is kept available for the embedding model and system needs. This setup allows both models to run smoothly on a single node.

In [ ]:
from langchain_community.llms import VLLM

project_path = "/dartfs/rc/nosnapshots/V/VaickusL-nb/EDIT_Students/users/JiQing/LLM Project"

print("Loading vLLM directly into notebook memory. This will allocate GPU VRAM...")

# Initialize vLLM inline. 
# We limit gpu_memory_utilization to 0.8, so it leaves room for our Chroma embeddings.
llm = VLLM(
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    trust_remote_code=True,  # Required for some modern HuggingFace models
    max_new_tokens=1024,
    temperature=0.0,         # Zero temperature for clinical accuracy
    vllm_kwargs={"gpu_memory_utilization": 0.8},
    download_dir = f"{project_path}/cache"
)

print("\nvLLM engine loaded successfully!")

By design, vLLM reserves almost all available GPU memory upfront (based on your *gpu_memory_utilization: 0.8* setting) to optimize the KV-cache.

When I re-run the above Cell in the Jupyter Notebook without properly shutting down the previous run, the old VLLM::EngineCore processes stay alive in the background, holding onto that 26.9GB of VRAM. When the new cell execution tries to start a new vLLM instance, it finds the GPU is full, fails to initialize, and throws that *Engine core initialization failed error*.

So I need to clear the GPU before running. In terminal:
*kill -9 1634243 1700459*

*1634243 1700459* are job IDs

## 4.2 Defining the Structured Output Schema
To satisfy the requirement for structured reasoning, we define a Pydantic schema. This ensures the LLM's response always breaks down the clinical scenario into exact, machine-readable fields (ex. Current Step, Next Action, Safety Warnings).

We will use *Optional* types and add a *query_intent* field. This forces the LLM to pause and categorize the question before generating the rest of the JSON.

__`SurgicalQAOutput(BaseModel)`:__

You can think of `BaseModel` as the core building block for how we define and control our data. It’s a class from the `pydantic` library that adds type checking and validation to our Python objects.

When our `SurgicalQAOutput` class is built on top of `BaseModel`, it turns into a strong validation layer. Its role is to make sure the data coming from the LLM matches exactly what we expect before the application uses it.

For example, if I define `safety_warnings` as a list of strings, but the model returns something incorrect—like a single number—BaseModel will catch the issue, raise a validation error, and prevent that data from reaching the UI or clinical dashboard.

In [15]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser

# Define a flexible, intent-driven structure
class SurgicalQAOutput(BaseModel):
    query_intent: str = Field(description="Classify the query into one of four categories: 'Surgical Step Understanding', 'Clinical Reasoning', 'Knowledge Retrieval', or 'Context-based'.")
    primary_answer: str = Field(description="A detailed explanation answering the core question, grounded ONLY in retrieved guidelines.")
    current_step: Optional[str] = Field(default=None, description="The current surgical step. ONLY output this if the query involves a specific point in the surgery. Otherwise, output null.")
    next_action: Optional[str] = Field(default=None, description="The immediate next step. ONLY output this if applicable to the query. Otherwise, output null.")
    safety_warnings: Optional[List[str]] = Field(default=None, description="Key risks or safety considerations. ONLY output if applicable to the query. Otherwise, output null.")

# Initialize the parser
output_parser = JsonOutputParser(pydantic_object=SurgicalQAOutput)

print("Flexible Pydantic schema and JSON parser initialized.")

Flexible Pydantic schema and JSON parser initialized.


## 4.3 Building the RAG Generation Pipeline
We now link our Phase 3 Retriever to our Phase 4 LLM. The prompt dynamically injects the retrieved clinical guidelines, the user's question, and the JSON formatting instructions.

In addition to structured output, the system uses a __Multi-step reasoning__ process. Early on, using a fixed schema led the model to generate incorrect surgical steps for general knowledge questions.

To address this, I designed an intent-aware prompt that guides the model through a step-by-step process. __Step 1:__ It reviews the user’s question and identifies the clinical intent. __Step 2:__ Decide which fields of the JSON structure are relevant based on that classification. __Step 3:__ It generates the final response.

This step-by-step approach helps prevent incorrect or misleading structured outputs.

__`PromptTemplate`:__

A `PromptTemplate` is a reusable way to structure how we communicate with the LLM.

Instead of building a long f-string every time a user asks a question, it lets us define the overall format once. It includes our main system instructions, like ‘You are an expert AI clinical assistant’, and then sets up clear input fields, such as `{context}` and `{question}`.

Here’s how it works: right before the model runs, the pipeline fills in those fields with the retrieved clinical guidelines into the `{context}` and the user’s question into the `{question}` slot. This keeps the prompt consistent every time and makes the model’s behavior much more predictable.

- __`partial_variables`:__ This is a very practical feature for managing prompt complexity. In our template, some variables change every time, like the user’s question and the retrieved context, while others stay the same, such as the JSON formatting rules generated by the Pydantic parser. The `partial_variables` argument lets us set those fixed formatting instructions in advance. By doing this, the JSON rules are already included in the prompt when it’s created. This simplifies the pipeline, because when a user asks a question, the system only needs to pass in the context and the question; the formatting rules are already part of the prompt.

- __`RunnablePassthrough`:__ This is a really useful feature in LangChain Expression Language. It helps manage how data moves through the pipeline. When a user enters a question, it starts as a simple string. But our `PromptTemplate` expects a dictionary with two keys: `context` and `question`. So we need to send that same input in two directions—one to the retriever to get relevant documents, and the other directly into the prompt as the user’s question. `RunnablePassthrough()` is designed for it. It takes the original input and forwards it unchanged into the pipeline. This allows us to run retrieval while keeping the original question intact, so both pieces come together correctly before being passed to the LLM.

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Define the Intent-Aware Prompt
prompt = PromptTemplate(
    template="""You are an expert AI clinical assistant specializing in laparoscopic cholecystectomy. 
    Answer the user's question using ONLY the provided clinical context.
    
    CRITICAL INSTRUCTIONS:
    1. First, classify the user's question type (e.g., Knowledge Retrieval vs. Surgical Step).
    2. Provide the main answer in the 'primary_answer' field.
    3. ONLY fill out 'current_step', 'next_action', and 'safety_warnings' if the question implies a specific point in the surgical workflow (e.g., dissecting Calot's triangle). 
    4. If the question is a general definition or protocol (like ERAS guidelines or CVS), you MUST set 'current_step', 'next_action', and 'safety_warnings' to null.
    
    Clinical Context:
    {context}
    
    User Question: {question}
    
    {format_instructions}
    
    Output purely the JSON object without any markdown wrapping or additional text.
    """,
    input_variables=["question", "context"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

# Helper function to format the retrieved documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Create the LangChain Expression Language (LCEL) Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | output_parser
)

print("Intent-aware RAG Pipeline linked and ready for inference.")

Intent-aware RAG Pipeline linked and ready for inference.


Retrieving the documents is just the first step; the system also needs to use them correctly without generating incorrect information. I designed the LangChain Expression Language chain pipeline to pass the retrieved content directly into the prompt.

LangChain Expression Language, or LCEL, may look a bit unusual at first because of the pipe syntax (`|`), but it’s actually a very structured, step-by-step data pipeline.

Here’s how a single query, like ‘What are the risks of dissecting Calot’s triangle?’, moves through the system, from the moment the user submits it to the final JSON output:

__Step 1: Input Split (Parallel Execution)__
When we call `rag_chain.invoke(user_query)`, the raw text enters the first block of the pipeline:
`{"context": retriever | format_docs, "question": RunnablePassthrough()}`.
The system immediately splits the data flow into two parallel paths:
- Question Path: `RunnablePassthrough()` takes the original text and passes it forward as the `question` variable.
- Context Path: The same text is sent to the `retriever` to fetch relevant documents.

__Step 2: Vector Search (Retrieval)__
This is where embedding happens. The retriever takes the input text and uses the `BGE-small` model to convert it into a 384-dimensional vector. It then sends that vector to ChromaDB, where Maximal Marginal Relevance (MMR) is applied.

Importantly, the database doesn’t return vectors to the LLM. Instead, it uses similarity scores to find the most relevant text sections/chunks and then returns the original, human-readable content linked to those sections/chunks. The `format_docs` function then combines those four sections/chunks into a single block of clinical context.

__Step 3: Prompt Injection__
At this stage, the pipeline combines both paths. We now have a dictionary that includes the `{context}`, which is the full text retrieved from ChromaDB, and the `{question}`.

This dictionary is then passed into the `PromptTemplate`. The template fills in these variables, along with the fixed Pydantic formatting instructions, to create the final prompt that is sent to the model.

__Step 4: LLM Generation__
The final formatted prompt is then sent to `vLLM`. The Llama-3 model processes the system instructions, the clinical context, and the user’s question. It computes attention weights and generates the response step by step, one token at a time.

Because of how the prompt is designed, the output is produced in a raw JSON format.

Because I clearly instruct the Llama-3 model to answer using only the provided clinical context, it relies on those retrieved guidelines as its main reference when reasoning through the {question} and completely bypasses its own pre-trained biases before generating the final JSON output.

__Step 5: Output Parsing (Final Validation Step)__
The raw string generated by vLLM is then sent to the `output_parser`. This is where Pydantic handles the validation.

The model may include extra conversational text or wrap the response in Markdown, like JSON tags. The parser removes that extra formatting, extracts the JSON content, and checks it against our Pydantic `BaseModel`.

It then converts the result into a clean, highly structured Python dictionary. This final, verified object is what gets displayed or passed to the clinical UI

By using LCEL, I don’t need to write custom Python if/else logic to manage all the transitions between steps. The pipe syntax moves data cleanly from the vector retrieval stage, into the prompt, then through the model on the GPU, and finally through the validator. This makes the entire RAG architecture easy to follow and ready for deployment.

## 4.4 Live Inference Test
Let's test the system with a complex, pseudo-visual reasoning question from the assignment prompt.

### Creating an assistant function

In [17]:
import json

def Clinical_Agent():
    print("=====================================================")
    print("⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized")
    print("Type 'exit' or 'quit' to end the session.")
    print("=====================================================\n")

    # Start an interactive CLI loop
    while True:
        # 1. Get user input
        user_input = input("\n🧑‍⚕️ Surgeon (You): ")
    
        # 2. Check for exit commands
        if user_input.lower() in ['exit', 'quit']:
            print("\nEnding session. Goodbye!")
            break
        
        # Skip empty inputs
        if not user_input.strip():
            continue
        
        print("\n🤖 AI Assistant is thinking and searching guidelines...")
    
        try:
            # 3. Invoke the RAG chain with the user's input
            result = rag_chain.invoke(user_input)
        
            # 4. Print the structured output beautifully
            print("\n--- Structured Clinical Output ---")
            print(json.dumps(result, indent=4))
        
        except Exception as e:
            print(f"\n❌ An error occurred: {e}")

### Question Category: Surgical Step Understanding

In [21]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the current step if the surgeon is dissecting Calot’s triangle?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The current step is the dissection of the hepatocystic triangle.",
    "current_step": "Dissection of the hepatocystic triangle",
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [22]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the next step after identifying the cystic duct and artery?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "After identifying the cystic duct and artery, the next step is to clip the cystic artery and divide it using hook scissors, taking care not to dislodge the proximal clips.",
    "current_step": "Step 3: Cystic artery is clipped and divided",
    "next_action": "Division of the cystic duct",
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Clinical Reasoning

In [23]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the key safety considerations during cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The key safety considerations during cholecystectomy include identifying the critical view of safety, maintaining a clear dissection plane, and avoiding excessive retraction. Additionally, surgeons should be aware of the risk of bile duct injury and take steps to minimize it, such as using a laparoscopic cholecystectomy technique that emphasizes the principles of safe cholecystectomy as highlighted by the Society of American Gastrointestinal and Endoscopic Surgeons (SAGES).",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [25]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the risks at the stage of cystic duct dissection?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The risks at the stage of cystic duct dissection are not explicitly mentioned in the provided clinical context. However, it is essential to identify and tape ligate the cystic duct prior to fundus-first dissection of the gallbladder to minimize the risk of bile duct injury.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Knowledge Retrieval

In [26]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What is the Critical View of Safety (CVS)?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "In patients undergoing laparoscopic cholecystectomy, the Critical View of Safety (CVS) is a technique used for anatomic identification of the cystic duct and artery.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


In [27]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  What are the ERAS recommendations for cholecystectomy?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                                 | 0/1…


--- Structured Clinical Output ---
{
    "query_intent": "Knowledge Retrieval",
    "primary_answer": "The ERAS recommendations for cholecystectomy include a multidisciplinary approach to patient care, with a focus on minimizing postoperative complications and improving patient outcomes. This includes preoperative optimization, intraoperative techniques, and postoperative care protocols.",
    "current_step": null,
    "next_action": null,
    "safety_warnings": null
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### Question Category: Context-based Question (Pseudo Visual Input)

In [19]:
Clinical_Agent()

⚕️  Laparoscopic Cholecystectomy AI Assistant Initialized
Type 'exit' or 'quit' to end the session.




🧑‍⚕️ Surgeon (You):  The gallbladder is retracted superiorly, and dissection is being performed around Calot’s triangle. What is the current step, what is the next step, and what are the risks?



🤖 AI Assistant is thinking and searching guidelines...


Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                                                        | 0/1 [00:00<?…


--- Structured Clinical Output ---
{
    "query_intent": "Surgical Step Understanding",
    "primary_answer": "The dissection begins by incising peritoneum along the edge of the gallbladder on both sides to open up the hepatocystic triangle.",
    "current_step": "Dissecting Calot's triangle",
    "next_action": "Continue dissecting the triangle to expose the cystic duct and artery",
    "safety_warnings": "Avoid energy use near the duodenum which can be adherent to the gallbladder"
}



🧑‍⚕️ Surgeon (You):  exit



Ending session. Goodbye!


### (Optional) Interactive UI Dashboard (`ipywidgets`)

As an alternative to the programmatic testing shown above, I have also engineered a lightweight, graphical User Interface directly within this Jupyter environment. 

While setting up a separate web framework is common in production, it adds extra network layers and delay, which aren’t needed for a local HPC demo.

To fulfill the UI requirement efficiently, this cell utilizes `ipywidgets` to generate a native, interactive dashboard. It allows us to dynamically test new clinical scenarios, ask follow-up reasoning questions, and evaluate the agent's intent-classification in real-time, completely bypassing the need to modify code or re-execute cells.

__`ipywidgets`__ allows you to build a sleek, interactive graphical interface directly inside the notebook cell.

In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [ ]:
def Clinical_Agent():
    # ==========================================
    # 1. Define the UI Components
    # ==========================================
    header = widgets.HTML("<h3>⚕️ Laparoscopic Cholecystectomy AI Assistant</h3><p>Enter your clinical query below to search the SAGES and ERAS guidelines.</p>")

    query_input = widgets.Textarea(
        value='',
        placeholder='e.g., "What is the current step if the surgeon is dissecting Calot’s triangle?"',
        description='🧑‍⚕️ Query:',
        layout=widgets.Layout(width='90%', height='80px')
    )

    submit_button = widgets.Button(
        description=' Ask Assistant',
        button_style='primary', # Makes the button blue
        icon='stethoscope',     # Adds a medical icon
        layout=widgets.Layout(width='200px', margin='10px 0px 10px 100px')
    )

    output_area = widgets.Output(layout=widgets.Layout(border='1px solid #d3d3d3', padding='10px', width='90%'))

    # ==========================================
    # 2. Define the Execution Logic
    # ==========================================
    def on_submit_clicked(b):
        with output_area:
            # Clear the previous output before showing the new one
            clear_output(wait=True)
        
            user_query = query_input.value.strip()
            if not user_query:
                print("⚠️ Please enter a valid question.")
                return

            print("🤖 AI Assistant is classifying intent and searching clinical guidelines...")
            
            try:
                # Invoke your Intent-Aware RAG chain
                result = rag_chain.invoke(user_query)
            
                # Print the structured output beautifully
                print("\n✅ Response Generated:\n")
                print(json.dumps(result, indent=4))
            
            except Exception as e:
                print(f"\n❌ An error occurred: {e}")

    # ==========================================
    # 3. Link and Display the UI
    # ==========================================
    submit_button.on_click(on_submit_clicked)

    # Display the elements vertically
    ui_layout = widgets.VBox([header, query_input, submit_button, output_area])
    display(ui_layout)